## KINDLE SENTIMENT ANALYSIS

About Dataset
Context This is a small subset of dataset of Book reviews from Amazon Kindle Store category.

Content 5-core dataset of product reviews from Amazon Kindle Store category from May 1996 - July 2014. Contains total of 982619 entries. Each reviewer has at least 5 reviews and each product has at least 5 reviews in this dataset. Columns

asin - ID of the product, like B000FA64PK
helpful - helpfulness rating of the review - example: 2/3.
overall - rating of the product.
reviewText - text of the review (heading).
reviewTime - time of the review (raw).
reviewerID - ID of the reviewer, like A3SPTOKDG7WBLN
reviewerName - name of the reviewer.
summary - summary of the review (description).
unixReviewTime - unix timestamp.
Acknowledgements This dataset is taken from Amazon product data, Julian McAuley, UCSD website. http://jmcauley.ucsd.edu/data/amazon/

License to the data files belong to them.

Inspiration

Sentiment analysis on reviews.
Understanding how people rate usefulness of a review/ What factors influence helpfulness of a review.
Fake reviews/ outliers.
Best rated product IDs, or similarity between products based on reviews alone (not the best idea ikr).
Any other interesting analysis

In [ ]:
import pandas as pd


In [ ]:
data = pd.read_csv('/content/all_kindle_review .csv', index_col=0)

In [ ]:
data.head()

,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


#### Removing unusual columns

In [ ]:
df = data[['reviewText', 'rating']]
df

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4
...,...,...
11995,Valentine cupid is a vampire- Jena and Ian ano...,4
11996,I have read all seven books in this series. Ap...,5
11997,This book really just wasn't my cuppa. The si...,3
11998,"tried to use it to charge my kindle, it didn't...",1


If rating > 3 : Good Sentiment,
else if rating <= 3 : Bad Sentiment..  

In [ ]:
df['rating'] = df['rating'].apply(lambda x:1 if x > 3 else 0 )

/tmp/ipykernel_795/3777331169.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['rating'] = df['rating'].apply(lambda x:1 if x > 3 else 0 )


In [ ]:
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",0
1,Great short read. I didn't want to put it dow...,1
2,I'll start by saying this is the first of four...,0
3,Aggie is Angela Lansbury who carries pocketboo...,0
4,I did not expect this type of book to be in li...,1


In [ ]:
#checking null count
df.isnull().sum()

,0
reviewText,0
rating,0


In [ ]:
#checking unique count for rating
df['rating'].value_counts()

,count
rating,
0,6000
1,6000


Perfectly balanced dataset, no requirement of upsampling

In [ ]:
## Preprocessing review text
df['reviewText'].head()

,reviewText
0,"Jace Rankin may be short, but he's nothing to ..."
1,Great short read. I didn't want to put it dow...
2,I'll start by saying this is the first of four...
3,Aggie is Angela Lansbury who carries pocketboo...
4,I did not expect this type of book to be in li...


In [ ]:
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
df['reviewText'] = df['reviewText'].apply(lambda x: x.lower())

/tmp/ipykernel_795/1671792364.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['reviewText'] = df['reviewText'].apply(lambda x: x.lower())


In [ ]:
from bs4 import BeautifulSoup

## Removing spl characters
df['reviewText'] = df['reviewText'].apply(lambda x: re.sub('[^a-z A-Z 0-9]', '', x))
## remove stopwords
df['reviewText'] = df['reviewText'].apply(lambda x:" ".join([y for y in x.split() if y not in stopwords.words('english')]))
## Remove url
df['reviewText']=df['reviewText'].apply(lambda x: re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '' , str(x)))
## Remove html tags
df['reviewText']=df['reviewText'].apply(lambda x: BeautifulSoup(x, 'html.parser').get_text())
## Remove any additional spaces
df['reviewText']=df['reviewText'].apply(lambda x: " ".join(x.split()))

/tmp/ipykernel_795/438354817.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['reviewText'] = df['reviewText'].apply(lambda x: re.sub('[^a-z A-Z 0-9]', '', x))
/tmp/ipykernel_795/438354817.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['reviewText'] = df['reviewText'].apply(lambda x:" ".join([y for y in x.split() if y not in stopwords.words('english')]))
/tmp/ipykernel_795/438354817.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc

In [ ]:
df.head()

,reviewText,rating
0,jace rankin may short hes nothing mess man hau...,0
1,great short read didnt want put read one sitti...,1
2,ill start saying first four books wasnt expect...,0
3,aggie angela lansbury carries pocketbooks inst...,0
4,expect type book library pleased find price right,1


In [ ]:
df.isnull().sum()

,0
reviewText,0
rating,0


In [ ]:
## Train test split
from sklearn.model_selection import train_test_split

In [ ]:
X = df['reviewText']
X

,reviewText
0,jace rankin may short hes nothing mess man hau...
1,great short read didnt want put read one sitti...
2,ill start saying first four books wasnt expect...
3,aggie angela lansbury carries pocketbooks inst...
4,expect type book library pleased find price right
...,...
11995,valentine cupid vampire jena ian another vampi...
11996,read seven books series apocalypticadventure o...
11997,book really wasnt cuppa situation man capturin...
11998,tried use charge kindle didnt even register ch...


In [ ]:
y = df['rating']
y

,rating
0,0
1,1
2,0
3,0
4,1
...,...
11995,1
11996,1
11997,0
11998,0


In [ ]:
X_train, X_test,y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 58.2 MB/s eta 0:00:00


In [ ]:
import gensim

In [ ]:
from gensim.models import KeyedVectors, Word2Vec

In [ ]:
import gensim.downloader as api

In [ ]:
wv = api.load('word2vec-google-news-300')

In [ ]:
wv['man']

array([ 0.32617188,  0.13085938,  0.03466797, -0.08300781,  0.08984375,
       -0.04125977, -0.19824219,  0.00689697,  0.14355469,  0.0019455 ,
        0.02880859, -0.25      , -0.08398438, -0.15136719, -0.10205078,
        0.04077148, -0.09765625,  0.05932617,  0.02978516, -0.10058594,
       -0.13085938,  0.001297  ,  0.02612305, -0.27148438,  0.06396484,
       -0.19140625, -0.078125  ,  0.25976562,  0.375     , -0.04541016,
        0.16210938,  0.13671875, -0.06396484, -0.02062988, -0.09667969,
        0.25390625,  0.24804688, -0.12695312,  0.07177734,  0.3203125 ,
        0.03149414, -0.03857422,  0.21191406, -0.00811768,  0.22265625,
       -0.13476562, -0.07617188,  0.01049805, -0.05175781,  0.03808594,
       -0.13378906,  0.125     ,  0.0559082 , -0.18261719,  0.08154297,
       -0.08447266, -0.07763672, -0.04345703,  0.08105469, -0.01092529,
        0.17480469,  0.30664062, -0.04321289, -0.01416016,  0.09082031,
       -0.00927734, -0.03442383, -0.11523438,  0.12451172, -0.02

In [ ]:
from gensim.utils import simple_preprocess
from nltk.tokenize import sent_tokenize
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
words = []
for i in X:
  sent = sent_tokenize(i)
  for j in sent:
    words.append(simple_preprocess(j))

In [ ]:
words

[['jace',
  'rankin',
  'may',
  'short',
  'hes',
  'nothing',
  'mess',
  'man',
  'hauled',
  'saloon',
  'undertaker',
  'knows',
  'hes',
  'famous',
  'bounty',
  'hunter',
  'oregon',
  'shot',
  'man',
  'saloon',
  'finished',
  'years',
  'long',
  'quest',
  'avenge',
  'sisters',
  'murder',
  'trying',
  'figure',
  'next',
  'snottynosed',
  'farm',
  'boy',
  'rescued',
  'gang',
  'bullies',
  'offers',
  'money',
  'kill',
  'man',
  'forced',
  'ranch',
  'reluctantly',
  'agrees',
  'bring',
  'man',
  'justice',
  'kill',
  'outright',
  'first',
  'needs',
  'tell',
  'sisters',
  'widower',
  'newskyla',
  'kyle',
  'springer',
  'bailey',
  'riding',
  'trails',
  'sleeping',
  'ground',
  'past',
  'month',
  'trying',
  'find',
  'jace',
  'wants',
  'revenge',
  'man',
  'killed',
  'husband',
  'took',
  'ranch',
  'amongst',
  'crimes',
  'shes',
  'keen',
  'detour',
  'jace',
  'wants',
  'take',
  'realizes',
  'shes',
  'options',
  'hides',
  'behind',


In [ ]:
model = gensim.models.Word2Vec(words, vector_size=200, epochs=10)

In [ ]:
def avgword2vec(doc):
  valid_words = [model.wv[word] for word in doc if word in model.wv.index_to_key]
  if len(valid_words) > 0:
    return np.mean(valid_words, axis=0)
  else:
    return np.zeros(model.vector_size)

In [ ]:
from tqdm import tqdm
import numpy as np

In [86]:
X = []
for i in tqdm(range(len(words))):
  X.append(avgword2vec(words[i]))

100%|██████████| 12000/12000 [00:32<00:00, 366.47it/s]


In [87]:
len(X)

12000

In [88]:
X[0].reshape(1,-1).shape

(1, 200)

In [89]:
X_n = np.array(X)          # shape (n, 200)
print(X_n.shape)

d = pd.DataFrame(X_n)

(12000, 200)


In [93]:
X_train, X_test,y_train, y_test = train_test_split(d, y, test_size=0.25, random_state=42)

In [95]:
from sklearn.ensemble import RandomForestClassifier

In [96]:
rnf = RandomForestClassifier(n_estimators=200, criterion = 'gini')
rnf.fit(X_train, y_train)

RandomForestClassifier(n_estimators=200)

In [97]:
y_pred = rnf.predict(X_test)

In [98]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [99]:
print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

0.7866666666666666
[[1188  313]
 [ 327 1172]]
              precision    recall  f1-score   support

           0       0.78      0.79      0.79      1501
           1       0.79      0.78      0.79      1499

    accuracy                           0.79      3000
   macro avg       0.79      0.79      0.79      3000
weighted avg       0.79      0.79      0.79      3000



In [100]:
def preprocess_and_embed(text, model, wv_model):
    # Lowercase the text
    text = text.lower()
    # Remove special characters
    text = re.sub('[^a-z A-Z 0-9]', '', text)
    # Remove stopwords
    text = " ".join([word for word in text.split() if word not in stopwords.words('english')])
    # Remove URLs and HTML tags (assuming these were cleaned already, but good for new input)
    text = re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '', text)
    text = BeautifulSoup(text, 'html.parser').get_text()
    # Remove any additional spaces
    text = " ".join(text.split())

    # Tokenize the text into words
    processed_words = simple_preprocess(text)

    # Get average word vector
    valid_words = [wv_model[word] for word in processed_words if word in wv_model.index_to_key]
    if len(valid_words) > 0:
        return np.mean(valid_words, axis=0)
    else:
        return np.zeros(wv_model.vector_size)


# Example of predicting a new review
new_review = "This book was absolutely amazing! I loved every single page and couldn't put it down."
processed_new_review = preprocess_and_embed(new_review, model, model.wv)

# Reshape for prediction (model expects a 2D array)
new_review_vector = processed_new_review.reshape(1, -1)

# Predict sentiment
prediction = rnf.predict(new_review_vector)

if prediction[0] == 1:
    print(f"The sentiment for the review '{new_review}' is: Positive")
else:
    print(f"The sentiment for the review '{new_review}' is: Negative")


new_review_negative = "Terrible product, completely useless and a waste of money."
processed_new_review_negative = preprocess_and_embed(new_review_negative, model, model.wv)
new_review_vector_negative = processed_new_review_negative.reshape(1, -1)
prediction_negative = rnf.predict(new_review_vector_negative)

if prediction_negative[0] == 1:
    print(f"The sentiment for the review '{new_review_negative}' is: Positive")
else:
    print(f"The sentiment for the review '{new_review_negative}' is: Negative")

The sentiment for the review 'This book was absolutely amazing! I loved every single page and couldn't put it down.' is: Positive
The sentiment for the review 'Terrible product, completely useless and a waste of money.' is: Negative
